# 01 · Two Frames in 3D

### Recap & why now
Project 3 flew a quadcopter squashed into a vertical plane: one tilt angle, one
rotation matrix, three degrees of freedom. Everything worked because a single number
described the orientation.

In three dimensions that convenience disappears. A drone can roll, pitch and yaw at
once, the order those happen in changes the answer, and the tidy angle description has
a hole in it. Before any of that, we need to be precise about **which way is which** —
and to write the convention down so it never has to be guessed at again.

### Learning objectives
1. Distinguish the **world frame** from the **body frame**, and say which one a vector lives in.
2. State the **ENU** and **FLU** conventions this project uses, and why announcing them matters.
3. Explain why thrust is always $[0, 0, T]$ in the body frame.
4. Draw a quadcopter at any orientation from its body geometry.
5. Read the X-configuration motor numbering used for the rest of the project.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

## 1 · Where things are, and which way they point

**World frame** — bolted to the ground. Waypoints live here, gravity points down here,
and it never moves.

**Body frame** — bolted to the drone. The nose, the four motors and the thrust
direction live here, and the whole frame rotates with the vehicle.

Every vector in this project belongs to one or the other, and mixing them is the most
common source of quadcopter bugs. A rotation is simply the machine that converts.

In [ ]:
fig = plt.figure(figsize=(11.5, 4.2))
ax = fig.add_subplot(121, projection="3d")
for j, (vec, name, col) in enumerate([([1, 0, 0], "x — East", "C3"),
                                      ([0, 1, 0], "y — North", "C2"),
                                      ([0, 0, 1], "z — Up", "C0")]):
    ax.quiver(0, 0, 0, *vec, color=col, lw=2.5, arrow_length_ratio=0.15)   # One arrow per world axis.
    ax.text(*(np.array(vec)*1.2), name, color=col, fontsize=9)
set_3d(ax, (-1.4, 1.4), (-1.4, 1.4), (-0.4, 1.4))
ax.set_title("World frame — ENU, fixed to the ground", fontsize=10)

ax2 = fig.add_subplot(122, projection="3d")
tilted = euler_to_quat(np.deg2rad(25), np.deg2rad(15), np.deg2rad(30))     # A drone leaning about.
draw_quad(ax2, [0, 0, 0], tilted, scale=2.5)
R = quat_to_rotmat(tilted)
for j, (name, col) in enumerate([("nose", "C3"), ("left", "C2"), ("thrust", "C0")]):
    ax2.quiver(0, 0, 0, *(R[:, j]*0.9), color=col, lw=2, arrow_length_ratio=0.15)
    ax2.text(*(R[:, j]*1.1), name, color=col, fontsize=9)
set_3d(ax2, (-1.4, 1.4), (-1.4, 1.4), (-0.4, 1.4))
ax2.set_title("Body frame — FLU, carried by the drone", fontsize=10)
plt.tight_layout(); plt.show()

## 2 · The convention, announced once

> **📐 Every equation and every line of code in this project uses:**
> **ENU** for the world — $x$ East, $y$ North, $z$ **Up** — so gravity is
> $[0, 0, -9.81]$ and level hover needs $T = mg$ along $+z$.
> **FLU** for the body — $x$ forward through the nose, $y$ left, $z$ up through the rotors.

The other common choice is **NED**: $x$ North, $y$ East, $z$ **Down**, gravity
positive, hover thrust negative. Autopilot firmware such as PX4 and ArduPilot works in
NED internally. Neither is more correct; what is definitely wrong is not saying which
one you mean.

In [ ]:
gravity_enu = np.array([0.0, 0.0, -g])             # Down is -z, so gravity is negative.
thrust_body = np.array([0.0, 0.0, PARAMS["m"]*g])  # Rotors push along body +z, always.

print("gravity in ENU        :", gravity_enu)
print("hover thrust in body  :", thrust_body, " <- this vector NEVER changes")
print("their sum, level      :", np.round(quat_to_rotmat([1, 0, 0, 0]) @ thrust_body + gravity_enu, 12))

for label, r, p_, y_ in [("level        ", 0, 0, 0),
                         ("roll 20°     ", 20, 0, 0),
                         ("pitch 20°    ", 0, 20, 0),
                         ("yaw 40°      ", 0, 0, 40)]:
    q = euler_to_quat(*np.deg2rad([r, p_, y_]))
    world = quat_to_rotmat(q) @ thrust_body        # The same body vector, seen from the world.
    print("  %s thrust in world %s   vertical share %3.0f%%" %
          (label, np.round(world, 2), 100*world[2]/np.linalg.norm(world)))

print("\nYaw changes nothing about the thrust direction — it spins the drone AROUND that axis.")
print("Only roll and pitch tilt it, and only a tilted drone can accelerate sideways.")

## 3 · One consequence worth flagging early

In an FLU body frame, $y$ points **left**. Apply the right-hand rule to a positive
rotation about $+y$ and the nose goes **down**, not up.

So in this project **a positive pitch angle tips the nose down and drives the drone
forward**. Aerospace texts using NED, where $y$ points right, get the opposite sign.
This is not an error to be fixed later; it is what our chosen convention produces, and
the next cell confirms it numerically rather than arguing about it.

In [ ]:
print("  rotation          thrust axis in world     the drone accelerates")
for label, r, p_ in [("+20° roll ", 20, 0), ("-20° roll ", -20, 0),
                     ("+20° pitch", 0, 20), ("-20° pitch", 0, -20)]:
    z_world = quat_to_rotmat(euler_to_quat(*np.deg2rad([r, p_, 0])))[:, 2]   # Column 2 is body z.
    direction = ("+x East" if z_world[0] > 0.01 else "-x West" if z_world[0] < -0.01 else
                 "+y North" if z_world[1] > 0.01 else "-y South")
    print("  %s %-24s %s" % (label, np.round(z_world, 3), direction))

print("\nPositive pitch -> thrust leans toward +x -> the drone flies EAST, nose down. Confirmed.")
print("Write the convention on the wall. Every sign error in the next nine notebooks starts here.")

## 4 · The X-configuration

```text
              x_body (nose)
                   ↑
        M2 ●       |       ● M1
         (CCW)     |     (CW)
                   |
   y_body ←────────●────────
    (left)         |
         (CW)      |     (CCW)
        M3 ●       |       ● M4
```

Four motors on the diagonals, each at distance $L$ from the hub, so each sits
$L/\sqrt{2}$ along both body axes. **Diagonal pairs spin the same way**: in level hover
the two clockwise rotors cancel the two counter-clockwise ones, so the drone does not
spin. Break that balance deliberately and you get yaw.

This numbering is fixed for the whole project. Notebook 07 turns it into a matrix.

In [ ]:
print("  motor   position in the body frame [m]     spin     arm length")
for i, (mp, sp) in enumerate(zip(MOTOR_POS, SPIN)):
    print("   M%d    %-32s %s %10.3f m" %
          (i+1, np.round(mp, 4), "CCW" if sp > 0 else "CW ", np.linalg.norm(mp)))

fig, ax = plt.subplots(figsize=(4.6, 4.4))
for i, mp in enumerate(MOTOR_POS):
    ax.plot([0, mp[0]], [0, mp[1]], color="0.4", lw=2.5)                   # The arm.
    circ = plt.Circle((mp[0], mp[1]), 0.06, fill=False, lw=2.2,
                      color="C3" if i in (0, 1) else "C0")                 # The rotor disc.
    ax.add_patch(circ)
    ax.text(mp[0]*1.6, mp[1]*1.4, "M%d\n%s" % (i+1, "CCW" if SPIN[i] > 0 else "CW"),
            ha="center", fontsize=9)
ax.arrow(0, 0, 0.13, 0, head_width=0.02, color="C2")
ax.text(0.16, 0, "nose", fontsize=9, va="center")
ax.set_xlim(-0.32, 0.32); ax.set_ylim(-0.28, 0.28); ax.set_aspect("equal")
ax.set_xlabel("body x [m]"); ax.set_ylabel("body y [m]"); ax.set_title("Seen from above")
plt.show()

## 🧪 Try it yourself

**E1.** The drone yaws to face North instead of East. Does its thrust vector change in
the world frame? Explain using the body-frame definition of thrust.

**E2.** Write `thrust_in_world(roll, pitch, yaw, T)` and find the roll angle at which
exactly half of the thrust points sideways. What fraction is left holding the drone up?

In [ ]:
# --- Solution E1 ---
for yaw_deg in (0, 90, 180):
    q = euler_to_quat(0, 0, np.deg2rad(yaw_deg))
    print("E1: yaw %3d° -> thrust in world %s" % (yaw_deg, np.round(quat_to_rotmat(q) @ thrust_body, 4)))
print("    Identical every time. Thrust is [0, 0, T] in the BODY frame and yaw rotates the drone")
print("    about that very axis, so the axis itself never moves. Yaw changes where the nose and")
print("    the camera point; it cannot produce a single newton of sideways force.")

# --- Solution E2 ---
def thrust_in_world(roll, pitch, yaw, T):
    """Rotate a body-frame thrust of magnitude T into world coordinates."""
    q = euler_to_quat(np.deg2rad(roll), np.deg2rad(pitch), np.deg2rad(yaw))
    return quat_to_rotmat(q) @ np.array([0.0, 0.0, T])

print("\nE2:  roll    sideways share   vertical share")
for deg in (0, 15, 30, 45, 60):
    F = thrust_in_world(deg, 0, 0, T_HOVER if False else PARAMS["m"]*g)
    print("    %4d° %14.0f%% %15.0f%%" % (deg, 100*abs(F[1])/np.linalg.norm(F), 100*F[2]/np.linalg.norm(F)))
print("    Exactly half goes sideways at 30°, because sin(30°) = 0.5 — and %.0f%% is still holding" %
      (100*np.cos(np.deg2rad(30))))
print("    the drone up, because cos(30°) = %.3f. Tilting spends vertical force, and Notebook 06" %
      np.cos(np.deg2rad(30)))
print("    turns that into the equations of motion.")

## 🚁 Mini-project: the drone that leans

Animate the quadcopter rolling, then pitching, then yawing, with the thrust arrow
along for the ride. Nothing is being simulated yet — we are only drawing poses — but
this is the picture to hold on to for the next nine notebooks.

In [ ]:
segments = [("roll ", np.linspace(0, 35, 30), 0), ("roll ", np.linspace(35, 0, 20), 0),
            ("pitch", np.linspace(0, 35, 30), 1), ("pitch", np.linspace(35, 0, 20), 1),
            ("yaw  ", np.linspace(0, 90, 40), 2), ("yaw  ", np.linspace(90, 0, 25), 2)]
poses = []
for name, sweep, axis in segments:
    for deg in sweep:
        rpy = [0.0, 0.0, 0.0]; rpy[axis] = np.deg2rad(deg)                 # Move one axis at a time.
        poses.append((name, euler_to_quat(*rpy)))

fig = plt.figure(figsize=(6.2, 5.4))
ax = fig.add_subplot(111, projection="3d")

def frame(k):
    ax.clear()
    name, q = poses[k]
    draw_quad(ax, [0, 0, 0], q, scale=2.2)
    for j, col in enumerate(["0.75", "0.75", "0.75"]):
        ax.quiver(0, 0, 0, *(np.eye(3)[:, j]*0.8), color=col, lw=1.2, arrow_length_ratio=0.15)
    set_3d(ax, (-1.2, 1.2), (-1.2, 1.2), (-0.8, 1.2))
    rpy = np.degrees(quat_to_euler(q))
    ax.set_title("%s   roll %5.1f°  pitch %5.1f°  yaw %5.1f°" % (name, *rpy), fontsize=10)
    ax.view_init(elev=22, azim=-60)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(poses), interval=55, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Frame conventions are the single most common source of
> integration bugs in robotics, which is why ROS publishes a standard (REP-103: ENU
> world, FLU body — exactly what we chose) and why every autopilot documents its own.
> A drone that flies to a mirrored position, a camera that reports obstacles behind the
> vehicle, an IMU whose axes disagree with the airframe: all the same bug, and all
> found by writing the convention down first.

**Where next.** We can draw an orientation but not yet compute with one. Notebook 02
builds the three elementary rotation matrices and discovers that in 3-D, order matters.